In [42]:
import requests
import xml.etree.ElementTree as ET
import folium
from geopy.distance import geodesic
import numpy as np


In [36]:
urls = {
    "022":"https://www.google.com/maps/d/u/0/kml?forcekml=1&mid=1HrPAkSl8YlIRKMDonhyxLVAjGzADgh8",
    "023":"https://www.google.com/maps/d/u/0/kml?forcekml=1&mid=1qqCQfAT7F9ni_e-F6HYyiSDailGfyes",
    "024":"https://www.google.com/maps/d/u/0/kml?forcekml=1&mid=1rtDO3oIja8P2h9v_nAfdK5PcP32W7K4",
    "041":"https://www.google.com/maps/d/u/0/kml?forcekml=1&mid=1YTSC5gURkVEP9xdwj_0AyJrW_fpsmB8",
    "042":"https://www.google.com/maps/d/u/0/kml?forcekml=1&mid=14SprkRS7awK-ymG2Ze_2xFrGXDPzsNk"
        }

In [49]:
def parse_kml(kml_text):
    NS = "http://www.opengis.net/kml/2.2"
    root = ET.fromstring(kml_text)

    results = []

    for placemark in root.iter(f"{{{NS}}}Placemark"):
        # LineStrings (route polyline)
        linestring = placemark.find(f".//{{{NS}}}LineString")
        if linestring is not None:
            coords_el = linestring.find(f"{{{NS}}}coordinates")
            if coords_el is not None:
                points = []
                for coord in coords_el.text.strip().split():
                    parts = coord.split(",")
                    lon, lat = float(parts[0]), float(parts[1])
                    points.append([lat,lon])
                results = points
            
    return results

In [82]:
coords_rutas = {}
for ruta, link in urls.items():
    response = requests.get(url=link)
    coords_rutas[ruta] = parse_kml(response.text)

In [83]:
def get_elevations_batch(coords):
    """
    coords: list of (lat, lon) tuples
    """
    url = "https://api.open-elevation.com/api/v1/lookup"
    payload = {
        "locations": [
            {"latitude": lat, "longitude": lon}
            for lat, lon in coords
        ]
    }
    response = requests.post(
        url,
        json=payload,
        headers={"Content-Type": "application/json", "Accept": "application/json"}
    )
    response.raise_for_status()
    return [r["elevation"] for r in response.json()["results"]]

In [ ]:
def get_slope_color(incline_deg: float) -> str:
    """Return color based on absolute slope angle."""
    abs_inc = abs(incline_deg)
    if abs_inc <= 5:
        return "green"
    elif abs_inc <= 10:
        return "orange"
    else:
        return "red"

for ruta, coords in coords_rutas.items():
    # Fetch elevations and append to each coord
    elevations = get_elevations_batch(coords)
    coords_3d = [c[:2] + [e] for c, e in zip(coords, elevations)]

    m = folium.Map(location=coords_3d[0][:2], zoom_start=13)

    for i in range(1, len(coords_3d)):
        prev, curr = coords_3d[i - 1], coords_3d[i]

        horizontal_dist = geodesic(prev[:2], curr[:2]).meters  
        vertical_diff   = curr[2] - prev[2]

        inclinacion = np.degrees(np.arctan2(vertical_diff, horizontal_dist))

        folium.PolyLine(
            locations=[prev[:2], curr[:2]],         
            color=get_slope_color(inclinacion),
            weight=4,
            opacity=0.8,
            tooltip=f"{inclinacion:.1f}°"
        ).add_to(m)

    m.save(f"mapas/ruta_{ruta}.html")